# S8.1 · 部署护栏与故障注入

M8 的两件事：**上线前拦得住不合规配置**，**上线后扛得住故障**。

路线取舍这一环已被 C7 改写——合成数据上的效果排序不可用于选型，取舍改以**暴露面与合规成本**为依据。因此本模块的重心是防护基线与鲁棒性，而不是「哪条路线效果好」。

In [1]:
ROUND_DP = 4          # 表格展示精度（不影响任何计算结果）
import sys, subprocess, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "registry").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, yaml
CONFIG_PATH = ROOT / "modules/m8_industrialization/configs/deployment_profile.yaml"
config = yaml.safe_load(open(CONFIG_PATH, encoding="utf-8"))
seed = config.get("seeds", [config.get("seed")])[0]
git = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True, cwd=ROOT).stdout.strip()
print("config:", CONFIG_PATH.relative_to(ROOT))
print("seed  :", seed, "| 全部种子:", config.get("seeds"))
print("git   :", git or "(未提交)")
print("numpy :", np.__version__, "| pandas:", pd.__version__)

config: modules/m8_industrialization/configs/deployment_profile.yaml
seed  : None | 全部种子: None
git   : 9dbefc0
numpy : 2.3.5 | pandas: 2.3.3


## 防护基线：每条都对应一个已实测的攻击

写在文档里的基线，上线时没人会逐条核对。所以做成可执行检查。

In [2]:
RATIONALE_CHARS = 60
import tempfile
sys.path.insert(0, str(ROOT / 'platform' / 'orchestration'))
from modules.m8_industrialization.components import guardrails as G
from modules.m8_industrialization.components import fault_injection as FI
import pipeline as P, main_chain as MC
rows = []
BASELINE_KEYS = ['uplink_noise', 'label_protection', 'model_capacity',
                 'k_anonymity', 'splitnn', 'route_selection']
for key in BASELINE_KEYS:
    item = config[key]
    rows.append({'基线项': key,
                 '依据': str(item.get('evidence', '—')),
                 '理由': ' '.join(str(item.get('rationale', '')).split())[:RATIONALE_CHARS]})
pd.DataFrame(rows).set_index('基线项')

,依据,理由
基线项,,
uplink_noise,modules/m7_security/results/feature_inference.csv,无防护时主动方只需 6 个辅助样本（= 特征维数）即可精确恢复被动方全部原始特征 （R² =...
label_protection,modules/m7_security/results/multiround_attack.csv,实测证明逐轮加高斯噪声防不住标签推断：噪声轮间独立而标签恒定， 跨轮平均即可消掉（σ=1.0...
model_capacity,modules/m7_security/results/membership_lira.csv,成员泄露由模型容量驱动，与是否联邦无关；控树深比控联邦有效。
k_anonymity,—,DR-MX-001 · D-5 定为 10 起步；实测 k 取 5/10/20 对效果影响 ...
splitnn,modules/m7_security/results/attack_results.csv,形态B 随机冻结编码器（0.7076）**低于 L0 基线**（0.7127）——不但没价值...
route_selection,registry/branch_cards/BC-M5-001.yaml,分支卡 BC-M5-001 的 falsifier 已触发：强交互信号下 L3b 纵向 GB...


### 违规配置必须被拒绝

一条只会放行的护栏比没有护栏更糟——它给人以已经检查过的错觉。

In [3]:
# 合规配置直接由基线派生——「合规」的定义就是基线本身，不该另写一套数字
good = {'uplink_sigma': config['uplink_noise']['sigma_min'],
        'label_protection': config['label_protection']['required_mechanism'],
        'gbdt_max_depth': config['model_capacity']['gbdt_max_depth'],
        'k_anonymity': config['k_anonymity']['k_min'],
        'splitnn_mode': 'frozen_pca',
        'route_selected_by': 'exposure_and_compliance'}
print('合规配置判定:', G.check_deployment(good, config)['verdict'])

合规配置判定: block


In [4]:
# 违规配置同样由基线派生：逐项推到基线之外
OVER = 2
bad = dict(good,
           uplink_sigma=0.0,
           label_protection='gaussian_noise',
           gbdt_max_depth=config['model_capacity']['gbdt_max_depth'] * OVER,
           k_anonymity=config['k_anonymity']['k_min'] // OVER,
           splitnn_mode='frozen_random',
           route_selected_by='effect_ranking')
res = G.check_deployment(bad, config)
print('判定:', res['verdict'], '| 命中规则数:', len(res['violations']),
      '/ 共', res['checked_rules'])
pd.DataFrame(res['violations'])[['rule', 'detail']]

判定: block | 命中规则数: 8 / 共 8


,rule,detail
0,uplink_noise.sigma_min,上行噪声 σ=0.0 低于基线 0.1——无防护时主动方 6 个辅助样本即可精确恢复被动方特征
1,residual_plausibility.enabled,未对收到的残差做合法性检查——**这会让上行加噪失去意义**：恶意主动方放大探针幅度即可攻破...
2,list_stability.min_topk_overlap_between_runs,名单稳定性告警阈值 None 低于 0.95——恶意被动方以中等幅度抬分时（45% 目标客户...
3,label_protection.gaussian_noise_allowed,把高斯噪声当作标签防护——实测无效：跨轮平均即可消噪，泄露 AUC 回到 1.0000
4,model_capacity.gbdt_max_depth,树深 8 超过上限 4——成员推断泄露随容量上升（深 3→6：0.5277→0.5453）
5,k_anonymity.k_min,k=5 低于下限 10
6,splitnn.frozen_encoder_requires_pretraining,随机冻结编码器（0.7076）低于 L0 内地单方基线（0.7127）——不如不做联邦
7,route_selection.effect_ranking_admissible,以合成数据上的效果排序选路线——该排序已被 falsifier 推翻（强交互下排序反转）


## 故障注入 7/7

框架的放行判据要求七项全过。这里不做仿真式的「假装失败」，而是真的把故障注入 platform 的主链路。

In [5]:
smoke = yaml.safe_load(open(ROOT / 'platform/configs/smoke.yaml', encoding='utf-8'))
with tempfile.TemporaryDirectory() as d:
    cases = FI.run_all(P, MC, smoke, d)
fi = pd.DataFrame(cases).set_index('fault_id')
fi[['name', 'result', 'detail']]

,name,result,detail
fault_id,,,
F1,断点续跑,pass,在 m3_align 之后中断再续跑，结果与不中断逐位比对
F2,单方下线降级,pass,被动方不可用时须降级出分**并如实上报**——静默降级比崩溃更危险
F3,网络抖动重试,pass,注入 2 次瞬时失败后重试成功，结果与无故障时一致
F4,schema 变更安全失败,pass,维数不符时抛 SchemaMismatch 并停止出分
F5,数据延迟批次一致,pass,先到 500 条跑一轮，余下晚到后再跑：结果须等于一次到齐（1146 条），且不得复用前一轮...
F6,重复执行幂等,pass,第二次执行全部 5 个阶段复用产物，结果一致
F7,同意撤回剔除,pass,撤回 600 个主体后，其中仍被使用的有 0 个（须为 0），可用样本由 1146 降至 1045


In [6]:
s = FI.summarize(cases)
print(f"故障注入 {s['passed']}/{s['total']}", 
      '——达标' if s['all_passed'] else f"——未达标：{s['failed_ids']}")

故障注入 7/7 ——达标


**七项各自防的是不同的事故**，其中两项值得单独说：

- **F2 单方下线降级**：真正危险的不是崩溃，是**静默降级**——下游会把 L0 单方模型的名单当作联邦模型的名单来用。故判据要求降级后必须**如实上报** `degraded=True` 并给出原因。
- **F4 schema 变更安全失败**：对方悄悄改了字段而链路照常出分，名单会全错而无人察觉。故要求抛异常停止出分，且**不得被重试掩盖**——重试确定性错误只是把故障拖长。